# FEM screening stack -- test suite

Executable specification of the FEM-facing modules on the `fem_demo` branch, organised
**one section per module, one subsection per function**. Every check is a `@case`: it runs
immediately, prints `PASS`/`FAIL`, and is tallied at the end (the last cell asserts that
nothing failed, so the notebook can be executed headlessly).

Shared fixtures live in the *Setup* cell and nowhere else:

| helper | purpose |
|---|---|
| `fresh()` | re-initialise `EXPLOG` + `CANVAS` on a new temp directory |
| `design(**over)` / `result(**over)` | a baseline SOI design / a baseline calculator result |
| `enter(**over)` / `run(cid, ref)` | enter a design through the tool, run its analysis |
| `fixed_calc` / `boom_calc` | deterministic and always-failing calculators, patched via `with calc(...)` |
| `ref_of(msg, label)` | pull an 8-char reference id out of a tool message |

The generic memory modules (`past_steps`, `history_log`) keep their own pytest files; this
notebook covers `fem_materials`, `fem_calculator`, `relational_frame`, `fem_explog`,
`disposition_messages`, `tools`, `live_visualizer`, and `run_gate`.

In [1]:
import contextlib, math, os, sys, tempfile, traceback
from pathlib import Path

import pandas as pd

REPO = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(REPO))

from src import var
from src import fem_materials as FM
from src import fem_calculator as FC
from src.relational_frame import RelationalFrame
from src.fem_explog import FemExplog, DESIGN_COLUMNS, RESULT_COLUMNS, METRIC_COLUMNS, HIDDEN_CANDIDATE_COLUMNS, derived_metrics
from src import disposition_messages as DM
from src import run_gate as G
from src.fem_calculator import CalculationError, CalculationTimeout, DesignNotViable
import src.tools as T
from src.myCANVAS import CANVAS

EXPLOG = T.EXPLOG
RESULTS = []          # (name, ok, message)

def case(fn):
    """Run `fn` now; record PASS/FAIL (AssertionError or any exception) without stopping the cell."""
    try:
        fn(); RESULTS.append((fn.__name__, True, "")); print(f"PASS  {fn.__name__}")
    except Exception as e:
        tb = traceback.format_exc().strip().splitlines()[-1]
        RESULTS.append((fn.__name__, False, tb)); print(f"FAIL  {fn.__name__}: {tb}")
    return fn

# ---------------------------------------------------------------- fixtures
_TMP = tempfile.TemporaryDirectory()

def fresh():
    """New empty EXPLOG + CANVAS on a fresh temp dir; returns the dir."""
    d = Path(tempfile.mkdtemp(dir=_TMP.name))
    EXPLOG.init(d, disposition_decisions=var.DISPOSITION_DECISIONS, target_coupling_length_um=20.0)
    CANVAS.set_working_directory(str(d))
    CANVAS.canvas.clear(); CANVAS.result_registry.clear(); CANVAS.curr_round_result_ids.clear()
    var.my_WORKING_DIRECTORY = str(d); var.startTime = __import__("time").time()
    var.reportName = var.reportId = var.reportGist = ""
    return d

def design(**over):
    d = {"gap_um": 0.4, "wg1_width_um": 0.45, "wg2_width_um": 0.46, "core_height_um": 0.22,
         "wavelength_um": 1.55, "core_material": "Si", "n_core": 3.4777,
         "clad_material": "SiO2", "n_clad": 1.444}
    d.update(over); return d

def result(**over):
    r = {k: 1.0 for k in RESULT_COLUMNS}
    r.update(n_eff_symmetric=2.371, n_eff_antisymmetric=2.349, coupling_length_um=35.31, max_power_transfer=0.99)
    r.update(over); return r

def fixed_calc(params, *a, **kw):
    """Deterministic: L_c scales with gap, so sorting tests are predictable."""
    return result(coupling_length_um=20.0 * params["gap_um"] / 0.4, max_power_transfer=0.97)

def boom_calc(params, *a, **kw):
    """A design fault: recorded against the candidate."""
    raise DesignNotViable("waveguide 1 has no guided mode at 1.55 um")

def env_fail_calc(params, *a, **kw):
    """An environment fault: must NOT be recorded against the candidate."""
    raise CalculationError("the solver process exited with code -11 without producing a result")

@contextlib.contextmanager
def calc(fn):
    """Patch the solver entry point the tool layer calls (the subprocess hop)."""
    old = T.solve_in_subprocess; T.solve_in_subprocess = fn
    try: yield
    finally: T.solve_in_subprocess = old

def ref_of(msg, label):
    tok = [t for t in msg.replace("\n", " ").split() if t.startswith(label + "=")]
    assert tok, f"no {label} in: {msg[:200]}"
    return tok[0].split("=", 1)[1].rstrip(".")

ENTER_BASE = dict(reason_or_hypothesis="baseline", gap_um=0.4, wg1_width_um=0.45, wg2_width_um=0.45,
                  wavelength_um=1.55, core_material="Si", clad_material="SiO2", design_ref="")

def enter(**over):
    """enter_design_in_log through the tool; returns (message, Message_ID or None)."""
    msg = T.enter_design_in_log.invoke({**ENTER_BASE, **over})
    return msg, (ref_of(msg, "Message_ID") if "Message_ID=" in msg else None)

def run(cid, ref, note="n"):
    return T.run_coupled_mode_analysis.invoke({"candidate_id": cid, "candidate_id_ref": ref, "note": note})

def cand(cid):
    return EXPLOG.relational_frame.candidates[cid].series

print("setup ok:", REPO)

/home/ziqiw/.conda/envs/agent-ursa/lib/python3.11/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


setup ok: /nfs/turbo/coe-venkvis/ziqiw-turbo/material_agent


## 1. `fem_materials`

### 1.1 `refractive_index`

In [2]:
@case
def interpolates_linearly_between_tabulated_endpoints():
    n13, n16 = FM.refractive_index("Si", 1.30), FM.refractive_index("Si", 1.60)
    assert n13 > n16                                        # normal dispersion
    assert math.isclose(FM.refractive_index("Si", 1.45), (n13 + n16) / 2, rel_tol=1e-12)

@case
def material_lookup_is_case_insensitive():
    assert FM.refractive_index("si", 1.55) == FM.refractive_index("Si", 1.55)

@case
def unknown_material_raises_with_the_known_list():
    try: FM.refractive_index("Unobtainium", 1.55); assert False
    except KeyError as e: assert "Known materials" in str(e) and "Si" in str(e)

@case
def core_indices_exceed_cladding_indices_across_the_window():
    for w in (1.3, 1.45, 1.6):
        assert min(FM.refractive_index(m, w) for m in ("Si", "SiN", "LiNbO3", "SU8")) \
             > max(FM.refractive_index(m, w) for m in ("SiO2", "Air"))

PASS  interpolates_linearly_between_tabulated_endpoints
PASS  material_lookup_is_case_insensitive
PASS  unknown_material_raises_with_the_known_list
PASS  core_indices_exceed_cladding_indices_across_the_window


### 1.2 `default_core_height_um`, `known_materials`, `materials_table`

In [3]:
@case
def core_materials_have_a_default_height_and_claddings_do_not():
    assert FM.default_core_height_um("Si") == 0.22
    assert FM.default_core_height_um("SiO2") is None

@case
def known_materials_lists_every_table_entry():
    assert set(FM.known_materials()) == {"Si", "SiN", "LiNbO3", "SU8", "SiO2", "Air"}

@case
def materials_table_has_one_row_per_material_and_one_column_per_wavelength():
    lines = FM.materials_table(wavelengths_um=(1.3, 1.55)).splitlines()
    assert len(lines) == 1 + len(FM.known_materials())
    assert "n@1.30um" in lines[0] and "n@1.55um" in lines[0] and "default_core_height_um" in lines[0]

PASS  core_materials_have_a_default_height_and_claddings_do_not
PASS  known_materials_lists_every_table_entry
PASS  materials_table_has_one_row_per_material_and_one_column_per_wavelength


## 2. `fem_calculator` (real FEMWell solve, run out-of-process)

### 2.1 `compute_coupled_modes`

In [4]:
import time as _time
_t = _time.time(); DOC = FC.compute_coupled_modes(design()); print(f"FEM run: {_time.time() - _t:.1f} s")

@case
def returns_every_result_key_as_a_finite_float():
    assert set(DOC) == set(FC.RESULT_KEYS) == set(RESULT_COLUMNS)
    assert all(isinstance(v, float) and math.isfinite(v) for v in DOC.values())

@case
def reproduces_the_femwell_example_numbers():
    # https://helgegehring.github.io/femwell/photonics/examples/coupled_mode_theory.html
    assert abs(DOC["n_eff_symmetric"] - 2.3711) < 2e-3 and abs(DOC["n_eff_antisymmetric"] - 2.3492) < 2e-3
    assert abs(DOC["n_eff_wg1"] - 2.3495) < 2e-3 and abs(DOC["n_eff_wg2"] - 2.3707) < 2e-3
    assert abs(DOC["coupling_length_um"] - 35.31) < 0.3

@case
def effective_indices_lie_between_cladding_and_core():
    d = design()
    for k in ("n_eff_wg1", "n_eff_wg2", "n_eff_symmetric", "n_eff_antisymmetric"):
        assert d["n_clad"] < DOC[k] < d["n_core"]
    assert DOC["n_eff_symmetric"] > DOC["n_eff_antisymmetric"]

@case
def coupling_length_is_the_supermode_beat_half_period():
    expected = design()["wavelength_um"] / (2 * (DOC["n_eff_symmetric"] - DOC["n_eff_antisymmetric"]))
    assert math.isclose(DOC["coupling_length_um"], expected, rel_tol=1e-9)
    # CMT and supermode pictures agree on the beat length
    beta_c = math.pi / (2 * DOC["coupling_length_um"])
    assert DOC["kappa_12_per_um"] < beta_c and DOC["kappa_21_per_um"] < beta_c   # detuning adds to beta_c

@case
def a_10nm_mismatch_detunes_the_cores_and_kills_power_transfer():
    assert 0 <= DOC["max_power_transfer"] < 0.15
    assert DOC["kappa_12_per_um"] > 0 and DOC["kappa_21_per_um"] > 0

@case
def invalid_designs_are_rejected_as_DesignNotViable_before_meshing():
    # The TYPE carries the meaning: a bad DESIGN, not a broken environment.
    for bad in (design(n_core=1.4, n_clad=1.5), design(gap_um=0)):
        try: FC.compute_coupled_modes(bad); assert False
        except DesignNotViable: pass
    try: FC.compute_coupled_modes({k: v for k, v in design().items() if k != "gap_um"}); assert False
    except DesignNotViable as e: assert "gap_um" in str(e)

@case
def the_exception_hierarchy_keeps_design_and_environment_faults_separable():
    assert issubclass(CalculationTimeout, CalculationError)
    assert not issubclass(DesignNotViable, CalculationError)
    assert not issubclass(CalculationError, DesignNotViable)

@case
def the_subprocess_hop_returns_the_same_numbers_as_in_process():
    # gmsh.initialize() installs a SIGINT handler, which CPython allows only on
    # the main thread -- and LangGraph runs tool calls on worker threads. The
    # solve therefore happens in a child process; the numbers must not change.
    out = FC.solve_in_subprocess(design())
    assert all(math.isclose(out[k], DOC[k], abs_tol=1e-9) for k in RESULT_COLUMNS)

@case
def a_child_that_dies_without_a_result_is_an_environment_error():
    class Dead:
        returncode = -11                      # e.g. SIGSEGV inside the C solver
        stdout, stderr = "", "Segmentation fault"
    real_run = FC.subprocess.run
    FC.subprocess.run = lambda *a, **k: Dead()
    try:
        try: FC.solve_in_subprocess(design()); assert False
        except CalculationError as e:
            assert "-11" in str(e) and not isinstance(e, DesignNotViable)
    finally:
        FC.subprocess.run = real_run

@case
def a_solve_that_overruns_its_limit_is_a_timeout():
    try: FC.solve_in_subprocess(design(), timeout_seconds=0.5); assert False
    except CalculationTimeout as e: assert "0.5 s limit" in str(e)

@case
def mesh_policy_pads_low_contrast_and_keeps_the_example_window_for_soi():
    soi = FC._mesh_policy(design()); su8 = FC._mesh_policy(design(core_material="SU8", n_core=1.575, core_height_um=1.0))
    assert soi["w_sim"] == 4.0 and soi["h_box"] == 1.0 and soi["core_res"] == 0.03
    assert su8["h_box"] == 3.0 and su8["w_sim"] > soi["w_sim"] and su8["core_res"] > soi["core_res"]

FEM run: 23.1 s
PASS  returns_every_result_key_as_a_finite_float
PASS  reproduces_the_femwell_example_numbers
PASS  effective_indices_lie_between_cladding_and_core
PASS  coupling_length_is_the_supermode_beat_half_period
PASS  a_10nm_mismatch_detunes_the_cores_and_kills_power_transfer
PASS  invalid_designs_are_rejected_as_DesignNotViable_before_meshing
PASS  the_exception_hierarchy_keeps_design_and_environment_faults_separable


PASS  the_subprocess_hop_returns_the_same_numbers_as_in_process
PASS  a_child_that_dies_without_a_result_is_an_environment_error


PASS  a_solve_that_overruns_its_limit_is_a_timeout
PASS  mesh_policy_pads_low_contrast_and_keeps_the_example_window_for_soi


## 3. `relational_frame`

### 3.1 `Table.add_row` / `update_row` / `__getitem__`

In [5]:
def _rf():
    rf = RelationalFrame(root="parents")
    rf.add_table("parents", pd.DataFrame({"pid": pd.Series(dtype="string"), "x": pd.Series(dtype="Float64")}), pk="pid")
    rf.add_table("kids", pd.DataFrame({"kid": pd.Series(dtype="Int64"), "pid": pd.Series(dtype="string")}), pk="kid")
    rf.parents.add_relation(name="kids", child_table="kids", fk_in_child="pid")
    return rf

@case
def add_row_then_read_back_through_rowview():
    rf = _rf(); rf.parents.add_row({"pid": "a", "x": 1.5})
    assert rf.parents["a"].x == 1.5 and len(rf.parents.df) == 1

@case
def duplicate_pk_needs_allow_update_and_then_updates_in_place():
    rf = _rf(); rf.parents.add_row({"pid": "a", "x": 1.0})
    try: rf.parents.add_row({"pid": "a", "x": 2.0}); assert False
    except ValueError: pass
    rf.parents.update_row("a", x=2.0)
    assert rf.parents["a"].x == 2.0 and len(rf.parents.df) == 1

@case
def unknown_column_is_rejected_unless_allowed():
    rf = _rf()
    try: rf.parents.add_row({"pid": "a", "y": 1}); assert False
    except ValueError: pass
    rf.parents.add_row({"pid": "a", "y": 1}, allow_new_columns=True)
    assert "y" in rf.parents.df.columns

@case
def missing_pk_raises_keyerror():
    rf = _rf()
    try: rf.parents["nope"]; assert False
    except KeyError: pass

PASS  add_row_then_read_back_through_rowview
PASS  duplicate_pk_needs_allow_update_and_then_updates_in_place
PASS  unknown_column_is_rejected_unless_allowed
PASS  missing_pk_raises_keyerror


### 3.2 `RelatedView` (relations) and `enforce_referential_integrity`

In [6]:
@case
def related_view_filters_children_and_autofills_the_fk():
    rf = _rf(); rf.parents.add_rows([{"pid": "a", "x": 1}, {"pid": "b", "x": 2}])
    rf.parents["a"].kids.add_row({"kid": 1}); rf.parents["a"].kids.add_row({"kid": 2}); rf.parents["b"].kids.add_row({"kid": 3})
    assert len(rf.parents["a"].kids) == 2 and rf.kids[3].pid == "b"
    assert [k.kid for k in rf.parents["a"].kids] == [1, 2]
    rf.enforce_referential_integrity()

@case
def broken_fk_is_detected():
    rf = _rf(); rf.parents.add_row({"pid": "a", "x": 1}); rf.kids.add_row({"kid": 1, "pid": "ghost"})
    try: rf.enforce_referential_integrity(); assert False
    except ValueError as e: assert "ghost" in str(e)

Referential integrity check passed.
PASS  related_view_filters_children_and_autofills_the_fk
PASS  broken_fk_is_detected


## 4. `fem_explog.FemExplog`

### 4.1 `init` -- schema

In [7]:
@case
def schema_has_design_result_progress_and_disposition_columns():
    fresh(); cdf, pdf = EXPLOG.relational_frame.candidates.df, EXPLOG.relational_frame.processes.df
    assert set(DESIGN_COLUMNS) <= set(cdf.columns) and set(RESULT_COLUMNS) <= set(cdf.columns)
    assert {"n_started", "n_finalized", "state", "decision", "needs_disposition_update", *HIDDEN_CANDIDATE_COLUMNS} <= set(cdf.columns)
    assert {"process_id", "candidate_id", "job_type", "status", "processNote", "error", *RESULT_COLUMNS} <= set(pdf.columns)
    assert len(cdf) == 0 and len(pdf) == 0

PASS  schema_has_design_result_progress_and_disposition_columns


### 4.2 `add_candidate` / `find_duplicate_design` / `design_of`

In [8]:
@case
def ids_are_chronological():
    fresh()
    assert [EXPLOG.add_candidate(design(gap_um=g), "h") for g in (0.2, 0.3, 0.4)] == ["dc_0001", "dc_0002", "dc_0003"]

@case
def exact_duplicate_is_refused_and_names_the_existing_id():
    fresh(); EXPLOG.add_candidate(design(), "h")
    try: EXPLOG.add_candidate(design(), "again"); assert False
    except ValueError as e: assert "dc_0001" in str(e)
    assert len(EXPLOG.relational_frame.candidates.df) == 1

@case
def duplicate_detection_ignores_material_case_but_not_geometry():
    fresh(); EXPLOG.add_candidate(design(), "h")
    assert EXPLOG.find_duplicate_design(design(core_material="si")) == "dc_0001"
    assert EXPLOG.find_duplicate_design(design(gap_um=0.4001)) is None

@case
def fresh_candidate_projection_is_zeroed_and_active():
    fresh(); cid = EXPLOG.add_candidate(design(), "h", notes="n")
    row = cand(cid)
    assert (int(row["n_started"]), int(row["n_finalized"]), row["state"]) == (0, 0, "active")
    assert pd.isna(row["coupling_length_um"]) and bool(row["needs_disposition_update"]) is False
    assert row["reason_or_hypothesis"] == "h" and row["notes"] == "n"

@case
def design_of_round_trips_the_inputs_as_python_types():
    fresh(); cid = EXPLOG.add_candidate(design(), "h"); d = EXPLOG.design_of(cid)
    assert set(d) == set(DESIGN_COLUMNS) and isinstance(d["gap_um"], float) and d["core_material"] == "Si"

PASS  ids_are_chronological


PASS  exact_duplicate_is_refused_and_names_the_existing_id


PASS  duplicate_detection_ignores_material_case_but_not_geometry


PASS  fresh_candidate_projection_is_zeroed_and_active
PASS  design_of_round_trips_the_inputs_as_python_types


### 4.3 `record_analysis` / `has_completed_analysis` / `_recompute_candidate_progress`

In [9]:
@case
def completed_analysis_is_recorded_and_copied_up():
    fresh(); cid = EXPLOG.add_candidate(design(), "h")
    pid = EXPLOG.record_analysis(cid, "note", result=result(), error=None)
    row = cand(cid); prow = EXPLOG.relational_frame.processes[pid].series
    assert pid == 0 and prow["status"] == "completed" and prow["processNote"] == "note"
    assert (int(row["n_started"]), int(row["n_finalized"]), row["state"]) == (1, 1, "active")
    assert math.isclose(float(row["coupling_length_um"]), 35.31)
    assert EXPLOG.has_completed_analysis(cid) == 0

@case
def failed_analysis_marks_failed_and_a_later_success_recovers():
    fresh(); cid = EXPLOG.add_candidate(design(), "h")
    pid = EXPLOG.record_analysis(cid, "n", result=None, error="boom")
    assert EXPLOG.relational_frame.processes[pid].series["error"] == "boom"
    assert cand(cid)["state"] == "failed" and EXPLOG.has_completed_analysis(cid) is None
    EXPLOG.record_analysis(cid, "n", result=result(), error=None)
    assert cand(cid)["state"] == "active" and int(cand(cid)["n_started"]) == 2

@case
def latest_completed_result_wins_the_copy_up():
    fresh(); cid = EXPLOG.add_candidate(design(), "h")
    EXPLOG.record_analysis(cid, "n", result=result(coupling_length_um=10.0), error=None)
    EXPLOG.record_analysis(cid, "n", result=result(coupling_length_um=30.0), error=None)
    assert float(cand(cid)["coupling_length_um"]) == 30.0

@case
def result_missing_a_key_is_rejected_before_writing():
    fresh(); cid = EXPLOG.add_candidate(design(), "h"); bad = result(); bad.pop("kappa_12_per_um")
    try: EXPLOG.record_analysis(cid, "n", result=bad, error=None); assert False
    except ValueError as e: assert "kappa_12_per_um" in str(e)
    assert len(EXPLOG.relational_frame.processes.df) == 0

@case
def process_ids_are_global_across_candidates():
    fresh(); a = EXPLOG.add_candidate(design(), "h"); b = EXPLOG.add_candidate(design(gap_um=0.5), "h")
    assert EXPLOG.record_analysis(a, "n", result=result(), error=None) == 0
    assert EXPLOG.record_analysis(b, "n", result=result(), error=None) == 1

PASS  completed_analysis_is_recorded_and_copied_up

PASS  failed_analysis_marks_failed_and_a_later_success_recovers


PASS  latest_completed_result_wins_the_copy_up
PASS  result_missing_a_key_is_rejected_before_writing


PASS  process_ids_are_global_across_candidates


### 4.4 `derived_metrics` -- the two ranking metrics (smaller is better)

In [10]:
@case
def metric_formulas():
    m = derived_metrics({"coupling_length_um": 35.31, "max_power_transfer": 0.5}, 20.0)
    assert math.isclose(m["coupling_length_deviation_um"], 15.31) and math.isclose(m["excess_loss_dB"], 3.0103, abs_tol=1e-3)
    assert derived_metrics({"coupling_length_um": 20.0, "max_power_transfer": 1.0}, 20.0) == {"coupling_length_deviation_um": 0.0, "excess_loss_dB": 0.0}
    assert math.isclose(derived_metrics({"coupling_length_um": 1, "max_power_transfer": 0.0}, 20.0)["excess_loss_dB"], 120.0)  # clamped, never inf

@case
def metrics_are_written_on_both_tables_and_follow_the_latest_result():
    fresh(); cid = EXPLOG.add_candidate(design(), "h")
    p0 = EXPLOG.record_analysis(cid, "n", result=result(coupling_length_um=35.31, max_power_transfer=0.5), error=None)
    assert math.isclose(float(EXPLOG.relational_frame.processes[p0].series["coupling_length_deviation_um"]), 15.31)
    assert math.isclose(float(cand(cid)["excess_loss_dB"]), 3.0103, abs_tol=1e-3)
    EXPLOG.record_analysis(cid, "n", result=result(coupling_length_um=19.0, max_power_transfer=1.0), error=None)
    assert math.isclose(float(cand(cid)["coupling_length_deviation_um"]), 1.0) and float(cand(cid)["excess_loss_dB"]) == 0.0
    pf = EXPLOG.record_analysis(cid, "n", result=None, error="x")
    assert pd.isna(EXPLOG.relational_frame.processes[pf].series["excess_loss_dB"])   # failures carry no metrics

@case
def metrics_self_heal_on_a_frame_that_predates_them():
    fresh(); cid = EXPLOG.add_candidate(design(), "h")
    EXPLOG.record_analysis(cid, "n", result=result(coupling_length_um=25.0), error=None)
    for tbl in (EXPLOG.relational_frame.candidates, EXPLOG.relational_frame.processes):
        tbl.df = tbl.df.drop(columns=list(METRIC_COLUMNS))
    EXPLOG.update_log()
    assert math.isclose(float(cand(cid)["coupling_length_deviation_um"]), 5.0)
    assert math.isclose(float(EXPLOG.relational_frame.processes[0].series["coupling_length_deviation_um"]), 5.0)

PASS  metric_formulas


PASS  metrics_are_written_on_both_tables_and_follow_the_latest_result
PASS  metrics_self_heal_on_a_frame_that_predates_them


### 4.5 `candidate_outstanding` / `candidates_needing_disposition` / `update_log`

In [11]:
@case
def outstanding_units_are_one_per_terminal_process():
    fresh(); cid = EXPLOG.add_candidate(design(), "h")
    assert EXPLOG.candidate_outstanding(cid) == {"must_cover": [], "latest_disposition": None, "has_finalized": False}
    p0 = EXPLOG.record_analysis(cid, "n", result=None, error="x"); p1 = EXPLOG.record_analysis(cid, "n", result=result(), error=None)
    out = EXPLOG.candidate_outstanding(cid)
    assert [u["ids"] for u in out["must_cover"]] == [[p0], [p1]] and out["has_finalized"]

@case
def needs_disposition_tracks_uncovered_finished_work():
    fresh(); a = EXPLOG.add_candidate(design(), "h"); b = EXPLOG.add_candidate(design(gap_um=0.5), "h")
    assert EXPLOG.candidates_needing_disposition() == []
    EXPLOG.record_analysis(b, "n", result=result(), error=None)
    assert EXPLOG.candidates_needing_disposition() == [b]
    assert EXPLOG.update_log() == {}                      # nothing async; interface kept

PASS  outstanding_units_are_one_per_terminal_process


PASS  needs_disposition_tracks_uncovered_finished_work


### 4.6 `get_disposition_info` / `update_disposition_info` -- every status path

In [12]:
def _dispositioned():
    fresh(); cid = EXPLOG.add_candidate(design(), "h")
    pid = EXPLOG.record_analysis(cid, "n", result=result(), error=None)
    return cid, pid

def _upd(cid, ids, decision="Medium priority"):
    return EXPLOG.update_disposition_info(cid, "analysis", ids, "plan", decision)

@case
def unknown_candidate_on_both_calls():
    fresh()
    assert EXPLOG.get_disposition_info("dc_9999") == {"unknown_candidate": True, "candidate_id": "dc_9999"}
    assert _upd("dc_9999", [])["status"] == "unknown_candidate"

@case
def write_is_locked_until_get_unlocks_it():
    cid, pid = _dispositioned()
    assert _upd(cid, [pid])["status"] == "locked"
    info = EXPLOG.get_disposition_info(cid)
    assert info["must_cover"] == [{"job_type": "coupled_mode_analysis", "ids": [pid]}]
    assert bool(cand(cid)["ready_for_disposition_update"]) is True

@case
def each_rejection_leaves_the_lock_open_and_writes_nothing():
    cid, pid = _dispositioned(); EXPLOG.get_disposition_info(cid)
    assert _upd(cid, [pid], "Investigating")["status"] == "invalid_decision"
    assert _upd(cid, [99])["status"] == "foreign_ids"
    r = _upd(cid, []); assert r["status"] == "incomplete" and r["missing"] == [pid]
    assert cand(cid)["disposition_record"] == [] and bool(cand(cid)["ready_for_disposition_update"]) is True

@case
def success_appends_record_sets_decision_and_clears_flags():
    cid, pid = _dispositioned(); EXPLOG.get_disposition_info(cid)
    r = _upd(cid, [pid], "High priority")
    assert r["ok"] and r["status"] == "ok"
    row = cand(cid)
    assert row["decision"] == "High priority" and row["disposition_record"][-1]["Summarized_process_id"] == [pid]
    assert bool(row["needs_disposition_update"]) is False and bool(row["ready_for_disposition_update"]) is False
    assert EXPLOG.candidate_outstanding(cid)["latest_disposition"]["Decision"] == "High priority"

@case
def new_finished_work_reopens_the_need_and_only_new_ids_must_be_cited():
    cid, pid = _dispositioned(); EXPLOG.get_disposition_info(cid); _upd(cid, [pid])
    pid2 = EXPLOG.record_analysis(cid, "n", result=result(), error=None)
    assert EXPLOG.candidates_needing_disposition() == [cid]
    assert EXPLOG.get_disposition_info(cid)["must_cover"] == [{"job_type": "coupled_mode_analysis", "ids": [pid2]}]
    assert _upd(cid, [pid2])["ok"] and len(cand(cid)["disposition_record"]) == 2

PASS  unknown_candidate_on_both_calls
PASS  write_is_locked_until_get_unlocks_it


PASS  each_rejection_leaves_the_lock_open_and_writes_nothing
PASS  success_appends_record_sets_decision_and_clears_flags


PASS  new_finished_work_reopens_the_need_and_only_new_ids_must_be_cited


### 4.7 `_save_pickle`

In [13]:
@case
def pickle_is_written_atomically_after_each_mutation():
    d = fresh(); EXPLOG.add_candidate(design(), "h")
    assert (d / "explog.pkl").exists() and not (d / "explog.pkl.tmp").exists()
    payload = pd.read_pickle(d / "explog.pkl")
    assert set(payload) == {"candidates_df", "processes_df"} and len(payload["candidates_df"]) == 1

PASS  pickle_is_written_atomically_after_each_mutation


## 5. `disposition_messages` (pure formatters)

### 5.1 `_unit_desc` / `format_get_disposition`

In [14]:
@case
def unit_desc_handles_single_and_batch_ids():
    assert DM._unit_desc({"job_type": "coupled_mode_analysis", "ids": [4]}) == "coupled-mode analysis: process id 4"
    assert DM._unit_desc({"job_type": "other", "ids": [1, 2]}) == "other: process ids 1, 2 (cite any one)"

@case
def get_message_unknown_candidate_does_not_read_like_success():
    msg = DM.format_get_disposition("ghost", {"unknown_candidate": True})
    assert "ghost" in msg and "experiment log" in msg.lower() and "no prior disposition" not in msg.lower()

@case
def get_message_lists_outstanding_ids_and_tells_how_to_cite():
    msg = DM.format_get_disposition("dc_0001", {"must_cover": [{"job_type": "coupled_mode_analysis", "ids": [42]}],
                                                "latest_disposition": None, "has_finalized": True})
    assert "No prior disposition" in msg and "process id 42" in msg and "Analyzed_process_id" in msg

@case
def get_message_renders_latest_disposition_and_the_nothing_outstanding_case():
    latest = {"Decision": "High priority", "Summary": "looks good", "Future_plan": "sweep gap"}
    msg = DM.format_get_disposition("dc_0001", {"must_cover": [], "latest_disposition": latest, "has_finalized": True})
    assert "High priority" in msg and "looks good" in msg and "sweep gap" in msg and "no finished results awaiting" in msg

PASS  unit_desc_handles_single_and_batch_ids
PASS  get_message_unknown_candidate_does_not_read_like_success
PASS  get_message_lists_outstanding_ids_and_tells_how_to_cite
PASS  get_message_renders_latest_disposition_and_the_nothing_outstanding_case


### 5.2 `format_update_disposition` -- one branch per status

In [15]:
ALLOWED = var.DISPOSITION_DECISIONS

@case
def update_message_per_status():
    f = lambda **r: DM.format_update_disposition("dc_0001", r, ALLOWED)
    assert "Recorded disposition" in f(status="ok", decision="Abandon") and "Abandon" in f(status="ok", decision="Abandon")
    assert "get_disposition_info('dc_0001')" in f(status="locked")
    m = f(status="invalid_decision", decision="Maybe"); assert "Maybe" in m and all(a in m for a in ALLOWED)
    assert "7, 8" in f(status="foreign_ids", ids=[7, 8])
    assert "not finished" in f(status="non_terminal_ids", ids=[3])
    assert "5, 6" in f(status="incomplete", missing=[5, 6]) and "not recorded" in f(status="incomplete", missing=[5, 6])
    assert "experiment log" in f(status="unknown_candidate").lower()
    assert "Unexpected" in f(status="???")

PASS  update_message_per_status


### 5.3 `evaluate_terminal_tag_gate` / `format_needs_disposition_reminder`

In [16]:
def _gate(decision, state, in_flight):
    return DM.evaluate_terminal_tag_gate(decision=decision, state=state, has_in_flight=in_flight,
        terminal_decisions=var.DISPOSITION_TERMINAL_DECISIONS, active_decisions=var.DISPOSITION_ACTIVE_DECISIONS)

@case
def failed_candidate_may_only_be_abandoned():
    m = _gate("Medium priority", "failed", False); assert m and "FAILED" in m and "Abandon" in m and "Medium priority" in m
    assert _gate("Abandon", "failed", True) is None

@case
def terminal_tag_requires_nothing_in_flight_and_offers_active_options():
    m = _gate("Sufficient", "active", True); assert m and "in flight" in m.lower() and all(a in m for a in var.DISPOSITION_ACTIVE_DECISIONS)
    assert _gate("Sufficient", "active", False) is None and _gate("Abandon", None, False) is None

@case
def active_priority_is_always_allowed_when_not_failed():
    assert _gate("Low priority", "active", True) is None

@case
def reminder_is_empty_for_no_ids_and_sorted_otherwise():
    assert DM.format_needs_disposition_reminder([]) == ""
    m = DM.format_needs_disposition_reminder(["dc_0003", "dc_0001"])
    assert "dc_0001, dc_0003" in m and "get_disposition_info" in m

PASS  failed_candidate_may_only_be_abandoned
PASS  terminal_tag_requires_nothing_in_flight_and_offers_active_options
PASS  active_priority_is_always_allowed_when_not_failed
PASS  reminder_is_empty_for_no_ids_and_sorted_otherwise


## 6. `tools` (LangChain tools, invoked directly; calculator patched)

### 6.1 `list_materials`

In [17]:
@case
def list_materials_returns_the_table_and_registers_an_artifact():
    fresh(); msg = T.list_materials.invoke({})
    ref = ref_of(msg, "Reference_ID")
    assert "n@1.55um" in msg and CANVAS.get_artifact(ref).tool_name == "list_materials"

PASS  list_materials_returns_the_table_and_registers_an_artifact


### 6.2 `enter_design_in_log`

In [18]:
@case
def happy_path_defaults_height_and_resolves_indices_with_provenance():
    fresh(); mat_ref = ref_of(T.list_materials.invoke({}), "Reference_ID")
    msg, ref = enter(design_ref=mat_ref)
    assert "dc_0001" in msg and ref
    row = cand("dc_0001")
    assert math.isclose(float(row["core_height_um"]), 0.22) and float(row["n_core"]) > float(row["n_clad"])
    assert CANVAS.get_artifact(ref).parent_result_ids == [mat_ref]

@case
def input_validation_rejections():
    fresh()
    assert enter(design_ref="nope1234")[0].startswith("Error: Reference ID")
    assert "positive" in enter(gap_um=0)[0]
    assert "outside the study window" in enter(wavelength_um=2.0)[0]
    assert "must exceed cladding" in enter(core_material="SiO2", clad_material="Si")[0]
    assert "Unknown core material" in enter(core_material="Unobtainium")[0]
    assert len(EXPLOG.relational_frame.candidates.df) == 0

@case
def explicit_indices_allow_off_library_materials_but_need_a_height():
    fresh()
    assert "core_height_um is required" in enter(core_material="Unobtainium", n_core=2.0)[0]
    msg, _ = enter(core_material="Unobtainium", n_core=2.0, core_height_um=0.3)
    assert "dc_0001" in msg and float(cand("dc_0001")["n_core"]) == 2.0

@case
def duplicate_design_is_not_entered_and_points_at_the_existing_id():
    fresh(); enter(); msg, ref = enter()
    assert msg.startswith("Not entered") and "dc_0001" in msg and ref is None
    assert len(EXPLOG.relational_frame.candidates.df) == 1

PASS  happy_path_defaults_height_and_resolves_indices_with_provenance
PASS  input_validation_rejections
PASS  explicit_indices_allow_off_library_materials_but_need_a_height
PASS  duplicate_design_is_not_entered_and_points_at_the_existing_id


### 6.3 `run_coupled_mode_analysis`

In [19]:
@case
def completed_run_records_process_and_chains_provenance():
    fresh(); _, ref = enter()
    with calc(fixed_calc): out = run("dc_0001", ref, note="baseline")
    assert "completed" in out and "coupling_length_um = 20" in out
    assert "coupling_length_deviation_um = 0" in out and "target 20 um" in out and "excess_loss_dB = 0.132" in out
    pdf = EXPLOG.relational_frame.processes.df
    assert len(pdf) == 1 and pdf.loc[0, "status"] == "completed" and pdf.loc[0, "processNote"] == "baseline"
    art = CANVAS.get_artifact(ref_of(out, "Result_ID"))
    assert art.parent_result_ids == [ref] and art.metadata["status"] == "completed"

@case
def a_completed_design_is_not_recomputed():
    fresh(); _, ref = enter()
    with calc(fixed_calc): run("dc_0001", ref); again = run("dc_0001", ref)
    assert "already has a completed analysis" in again and len(EXPLOG.relational_frame.processes.df) == 1

@case
def an_unviable_design_is_recorded_and_may_be_retried():
    fresh(); _, ref = enter()
    with calc(boom_calc): out = run("dc_0001", ref)
    assert "NOT VIABLE" in out and "no guided mode" in out and cand("dc_0001")["state"] == "failed"
    with calc(fixed_calc): out2 = run("dc_0001", ref)
    assert "completed" in out2 and cand("dc_0001")["state"] == "active"

@case
def an_environment_fault_records_nothing_against_the_design():
    # The 2026-10-03 regression: a threading fault became nine "failed" designs,
    # which the terminal-tag gate then forced the agent to Abandon.
    fresh(); _, ref = enter()
    with calc(env_fail_calc): out = run("dc_0001", ref)
    assert "could NOT BE RUN" in out and "do NOT disposition it as" in out and "NOT VIABLE" not in out
    assert len(EXPLOG.relational_frame.processes.df) == 0
    assert cand("dc_0001")["state"] == "active"
    assert not bool(cand("dc_0001")["needs_disposition_update"])
    with calc(fixed_calc): assert "completed" in run("dc_0001", ref)   # retryable

@case
def bad_inputs_are_reported():
    fresh(); _, ref = enter()
    assert run("dc_0001", "nope1234").startswith("Error: Reference ID")
    assert run("dc_0099", ref).startswith("Error: no candidate")

@case
def reminder_names_other_undispositioned_designs_only():
    fresh(); _, r1 = enter(gap_um=0.2); _, r2 = enter(gap_um=0.4)
    with calc(fixed_calc): run("dc_0001", r1); out2 = run("dc_0002", r2)
    reminder = out2.split("REMINDER")[1].split("\n")[0]
    assert "dc_0001" in reminder and "dc_0002" not in reminder

PASS  completed_run_records_process_and_chains_provenance


PASS  a_completed_design_is_not_recomputed


PASS  an_unviable_design_is_recorded_and_may_be_retried
PASS  an_environment_fault_records_nothing_against_the_design


PASS  bad_inputs_are_reported


PASS  reminder_names_other_undispositioned_designs_only

### 6.4 `query_explog` / `read_explog`

In [20]:
def _two_designs_computed():
    fresh(); _, r1 = enter(gap_um=0.2); _, r2 = enter(gap_um=0.4)
    with calc(fixed_calc): run("dc_0001", r1); run("dc_0002", r2)

@case
def query_filters_sorts_paginates_and_hides_internal_columns():
    _two_designs_computed()
    q = T.query_explog.invoke({"table_name": "candidates", "reason": "r",
        "filters": [{"column": "n_started", "op": "ge", "value": 1}],
        "sort": [{"column": "coupling_length_um", "ascending": False}]})
    assert q.index("dc_0002") < q.index("dc_0001")                       # 20 > 10
    assert "Showing rows 0-1 of 2 total" in q
    best = T.query_explog.invoke({"table_name": "candidates", "reason": "rank",
        "sort": [{"column": "coupling_length_deviation_um", "ascending": True}]})
    assert best.index("dc_0002") < best.index("dc_0001")                  # |20-20| = 0 beats |10-20| = 10
    assert not any(c in q for c in HIDDEN_CANDIDATE_COLUMNS)
    assert "Query_result_ID=" in q

@case
def query_processes_table_and_pagination_hint():
    _two_designs_computed()
    p = T.query_explog.invoke({"table_name": "processes", "reason": "r", "start_idx": 1})
    assert "Showing rows 1-1 of 2 total" in p and "coupled_mode_analysis" in p
    assert T.query_explog.invoke({"table_name": "nope", "reason": "r"}).startswith("table_name must be")

@case
def query_with_an_unknown_column_surfaces_a_clear_error():
    _two_designs_computed()
    try: T.query_explog.invoke({"table_name": "candidates", "reason": "r", "filters": [{"column": "G(O)", "op": "eq", "value": 1}]}); assert False
    except Exception as e: assert "Unknown column" in str(e)

@case
def read_explog_shows_design_processes_and_latest_disposition():
    _two_designs_computed()
    T.get_disposition_info.invoke({"candidate_id": "dc_0001"})
    T.update_disposition_info.invoke({"candidate_id": "dc_0001", "Analysis_and_implications": "short L_c",
        "Analyzed_process_id": [0], "Future_plan": "widen gap", "Decision": "Low priority"})
    r = T.read_explog.invoke({"candidate_id": "dc_0001", "reasons": "check"})
    assert "gap_um" in r and "coupled_mode_analysis" in r and "Latest disposition" in r and "widen gap" in r
    assert "No candidate" in T.read_explog.invoke({"candidate_id": "dc_0099", "reasons": "x"})

PASS  query_filters_sorts_paginates_and_hides_internal_columns


PASS  query_processes_table_and_pagination_hint


PASS  query_with_an_unknown_column_surfaces_a_clear_error


PASS  read_explog_shows_design_processes_and_latest_disposition


### 6.5 `get_disposition_info` / `update_disposition_info` (tool layer adds the terminal-tag gate)

In [21]:
@case
def tool_layer_rejects_non_abandon_on_a_failed_design_then_accepts_abandon():
    fresh(); _, ref = enter()
    with calc(boom_calc): run("dc_0001", ref)
    g = T.get_disposition_info.invoke({"candidate_id": "dc_0001"}); assert "process id 0" in g and "Message_ID=" in g
    upd = lambda d: T.update_disposition_info.invoke({"candidate_id": "dc_0001", "Analysis_and_implications": "a",
                                                      "Analyzed_process_id": [0], "Future_plan": "f", "Decision": d})
    rej = upd("Medium priority"); assert "FAILED" in rej and "Abandon" in rej
    assert cand("dc_0001")["disposition_record"] == []                    # gate rejection wrote nothing
    assert "Recorded disposition" in upd("Abandon") and cand("dc_0001")["decision"] == "Abandon"

@case
def disposition_readout_lists_the_outstanding_results_with_metrics():
    fresh(); _, ref = enter(gap_um=0.2)
    assert "Results awaiting analysis" not in T.get_disposition_info.invoke({"candidate_id": "dc_0001"})
    with calc(fixed_calc): run("dc_0001", ref)
    g = T.get_disposition_info.invoke({"candidate_id": "dc_0001"})
    assert "Results awaiting analysis" in g and "target coupling length 20 um" in g
    assert "coupling_length_deviation_um" in g and "excess_loss_dB" in g and "10.0" in g   # |10 - 20|

@case
def tool_layer_passes_engine_statuses_through():
    fresh(); _, ref = enter()
    with calc(fixed_calc): run("dc_0001", ref)
    locked = T.update_disposition_info.invoke({"candidate_id": "dc_0001", "Analysis_and_implications": "a",
                                               "Analyzed_process_id": [0], "Future_plan": "f", "Decision": "Sufficient"})
    assert "get_disposition_info('dc_0001')" in locked
    assert "No candidate" in T.get_disposition_info.invoke({"candidate_id": "dc_0404"})

PASS  tool_layer_rejects_non_abandon_on_a_failed_design_then_accepts_abandon
PASS  disposition_readout_lists_the_outstanding_results_with_metrics
PASS  tool_layer_passes_engine_statuses_through


### 6.6 Canvas tools: `inspect_my_canvas` / `write_my_canvas` / `read_my_canvas`

In [22]:
@case
def write_read_inspect_round_trip_and_overwrite_guard():
    fresh()
    assert "successfully added" in T.write_my_canvas.invoke({"key": "k", "value": {"a": 1}})
    assert "already exists" in T.write_my_canvas.invoke({"key": "k", "value": 2})
    assert "successfully overwritten" in T.write_my_canvas.invoke({"key": "k", "value": 2, "overwrite": True})
    assert T.read_my_canvas.invoke({"key": "k"}) == 2 and "k" in T.inspect_my_canvas.invoke({})
    assert "not found" in T.read_my_canvas.invoke({"key": "zzz"})

##################### CANVAS #######################
{
'k': {
    'a': 1,
    }
}
################### CANVAS END #####################
##################### CANVAS #######################
{
'k': 2,
}
################### CANVAS END #####################
PASS  write_read_inspect_round_trip_and_overwrite_guard


### 6.7 `write_report`

In [23]:
@case
def report_requires_known_refs_never_overwrites_and_flags_compaction():
    fresh(); ref = ref_of(T.list_materials.invoke({}), "Reference_ID")
    args = {"report": "# r", "report_name": "round1", "report_gist": "gist", "ref_list": [ref]}
    assert T.write_report.invoke({**args, "ref_list": ["nope1234"]}).startswith("Error: Reference ID")
    out = T.write_report.invoke(args)
    assert "Report_ID=" in out.replace(": ", "=") and CANVAS.canvas["round1"] == "# r"
    assert (var.reportName, var.reportGist) == ("round1", "gist") and var.reportId
    assert T.write_report.invoke(args).startswith("ERROR: no report was written")

##################### CANVAS #######################
{
'round1': '# r',
}
################### CANVAS END #####################
PASS  report_requires_known_refs_never_overwrites_and_flags_compaction


### 6.8 `search_artifacts` / `check_time` / `format_time_budget`

In [24]:
@case
def search_by_tool_name_by_number_and_by_term_list():
    fresh(); _, ref = enter(gap_um=0.37)
    with calc(fixed_calc): run("dc_0001", ref)
    by_tool = T.search_artifacts.invoke({"query": None, "filters": {"tool_name": "enter_design_in_log"}, "order": "descending"})
    assert len(by_tool["results"]) == 1 and by_tool["results"][0]["result_id"] == ref
    by_num = T.search_artifacts.invoke({"query": 0.37, "filters": None, "order": "ascending"})
    assert any(r["result_id"] == ref for r in by_num["results"])
    by_terms = T.search_artifacts.invoke({"query": ["gap_um", "0.37"], "filters": None, "order": "descending"})
    assert any(r["result_id"] == ref for r in by_terms["results"])
    assert "Unknown filter category" in T.search_artifacts.invoke({"query": None, "filters": {"bogus": "x"}, "order": "descending"})

@case
def search_indexes_dict_valued_run_results_by_key_and_number():
    fresh(); _, ref = enter()
    with calc(fixed_calc): out = run("dc_0001", ref)
    run_ref = ref_of(out, "Result_ID")
    hit = T.search_artifacts.invoke({"query": ["excess_loss_dB", "0.13"], "filters": {"tool_name": "run_coupled_mode_analysis"}, "order": "descending"})
    assert [r["result_id"] for r in hit["results"]] == [run_ref]
    hit = T.search_artifacts.invoke({"query": 20.0, "filters": {"tool_name": "run_coupled_mode_analysis"}, "order": "descending"})
    assert [r["result_id"] for r in hit["results"]] == [run_ref]

@case
def time_budget_reports_remaining_and_percent_used():
    f = T.format_time_budget
    assert "No time budget is set" in f(300, None) and "No time budget is set" in f(300, 0)
    under = f(900, 1800)
    assert "0:15:00 of the 0:30:00 budget" in under and "50.0% used" in under
    assert "0:15:00 remaining" in under and "nearly exhausted" not in under

@case
def time_budget_warns_near_the_limit_and_reports_overrun():
    f = T.format_time_budget
    warn = f(1440, 1800)                                  # 80 % -- the warn threshold
    assert "80.0% used" in warn and "nearly exhausted" in warn
    assert "nearly exhausted" in f(1800, 1800)            # exactly at the limit
    over = f(2400, 1800)
    assert "OVER BUDGET by 0:10:00" in over and "133.3% used" in over
    assert "Do NOT start new designs" in over and "remaining" not in over
    # nothing is ENFORCED: the message is advice, the study keeps running
    assert "0:00:00" in f(-5, 1800) and "0:00:00" in f(None, 1800)

@case
def check_time_reads_the_live_budget_and_survives_an_unstarted_clock():
    fresh(); var.STUDY_BUDGET_SECONDS = 1800
    var.startTime = __import__("time").time() - 900
    out = T.check_time.invoke({})
    assert "of the 0:30:00 budget" in out and "50.0% used" in out
    var.startTime = ""
    assert "0:00:00" in T.check_time.invoke({})

query is None → matcher None
filters are {'tool_name': 'enter_design_in_log'} → matchers {'tool_name': {'kind': 'str', 'value': 'enter_design_in_log', 'str_form': 'enter_design_in_log'}}
{
  "n_matches": 1,
  "n_returned": 1,
  "order": "descending",
  "results": [
    {
      "result_id": "pronrhwx",
      "tool_name": "enter_design_in_log",
      "value_preview": "'dc_0001'",
      "args": {
        "candidate_id": "dc_0001",
        "gap_um": 0.37,
        "wg1_width_um": 0.45,
        "wg2_width_um": 0.45,
        "core_height_um": 0.22,
        "wavelength_um": 1.55,
        "core_material": "Si",
        "n_core": 3.4756,
        "clad_material": "SiO2",
        "n_clad": 1.4436666666666667,
        "reason_or_hypothesis": "baseline",
        "note": null
      },
      "description": "Entry of design dc_0001 into the experiment log with reason: baseline.",
      "context": ""
    }
  ]
}query is 0.37 → matcher {'kind': 'float', 'value': 0.37, 'float_value': 0.37}
filters are Non

## 7. `live_visualizer` (browser payload for the FEM explog + agent topology)

### 7.1 `_collect_explog` / `_flush`

In [25]:
import json as _json
from langchain_core.messages import AIMessage, ToolMessage
from src.live_visualizer import LiveVisualizer

def viz_payload(d):
    js = (Path(d) / "live_data.js").read_text()
    return _json.loads(js[js.index("{"): js.rindex(";\nwindow.__LIVE_TICK__")])

def viz_round(viz):
    """One full round as planNexe2.print_stream delivers it (node key already unwrapped)."""
    ev = lambda m: viz.on_event({"messages": [m]})
    def structured(name, args, tid, inp, outp):
        ev(AIMessage(content="", tool_calls=[{"name": name, "args": args, "id": tid}],
                     usage_metadata={"input_tokens": inp, "output_tokens": outp, "total_tokens": inp + outp}))
        ev(ToolMessage(content="ok", tool_call_id=tid, name=name))
    structured("Act", {"action": {"kind": "plan", "steps": [{"step": "Sweep the gap", "agent": "FEM_Agent", "required_tools": []}]}}, "t1", 1200, 300)
    ev(AIMessage(content="running", tool_calls=[{"name": "run_coupled_mode_analysis", "args": {"candidate_id": "dc_0001"}, "id": "t2"}],
                 usage_metadata={"input_tokens": 900, "output_tokens": 120, "total_tokens": 1020}))
    ev(ToolMessage(content="completed", tool_call_id="t2", name="run_coupled_mode_analysis"))
    structured("wokerResponse", {"answer": "a", "summary": "s", "success": True}, "t3", 1000, 80)
    structured("BossReview", {"decision": "approve", "feedback": ""}, "t4", 500, 40)
    viz.close()

def viz_fixture():
    """Populated explog + a visualizer pointed at a fresh dir."""
    d = fresh(); cid = EXPLOG.add_candidate(design(), "baseline")
    pid = EXPLOG.record_analysis(cid, "n", result=result(coupling_length_um=23.4, max_power_transfer=0.97), error=None)
    EXPLOG.get_disposition_info(cid)
    EXPLOG.update_disposition_info(cid, "SECRETTEXT " * 50, [pid], "sweep", "High priority")
    v = LiveVisualizer(canvas_obj=CANVAS, explog_obj=EXPLOG); v.set_working_directory(str(d))
    return d, v

@case
def explog_tables_reach_the_browser_with_the_metrics():
    d, v = viz_fixture(); data = viz_payload(d)
    assert data["explog"].get("_error") is None
    c, p = data["explog"]["candidates"][0], data["explog"]["processes"][0]
    assert all(isinstance(c[k], float) and isinstance(p[k], float) for k in METRIC_COLUMNS)
    assert math.isclose(c["coupling_length_deviation_um"], 3.4) and c["decision"] == "High priority"
    assert all(isinstance(x, (str, int, float, bool, type(None))) for x in c.values())   # JSON-native

@case
def agent_internal_columns_never_reach_the_browser():
    d, v = viz_fixture()
    c = viz_payload(d)["explog"]["candidates"][0]
    assert not [k for k in c if k in ("disposition_record", "ready_for_disposition_update")]
    assert "SECRETTEXT" not in (Path(d) / "live_data.js").read_text()
    assert "SECRETTEXT" not in (Path(d) / "live_visualization.html").read_text()

@case
def a_broken_explog_degrades_instead_of_killing_the_run():
    class Broken:
        @property
        def relational_frame(self): raise RuntimeError("explog exploded")
    d = fresh(); v = LiveVisualizer(explog_obj=Broken()); v.set_working_directory(str(d))
    assert "explog exploded" in viz_payload(d)["explog"]["_error"]

PASS  explog_tables_reach_the_browser_with_the_metrics


PASS  agent_internal_columns_never_reach_the_browser
PASS  a_broken_explog_degrades_instead_of_killing_the_run


### 7.2 step attribution and resume

In [26]:
@case
def each_step_is_attributed_to_the_agent_that_produced_it():
    d, v = viz_fixture(); viz_round(v); data = viz_payload(d)
    assert [(s["agent"], s["structured_response"]["type"]) for s in data["steps"]] == [
        ("Supervisor", "Act"), ("FEM_Agent", "wokerResponse"), ("Boss", "BossReview")]
    assert data["steps"][1]["title"].startswith("FEM_Agent")
    assert data["tokens_total"] == {"input": 3600, "output": 540}

@case
def the_worker_name_is_learned_from_the_plan_not_hard_coded():
    # planNexe2 unwraps the {node: update} envelope, so the worker's node name
    # only reaches the visualizer through the supervisor's plan steps.
    d, v = viz_fixture(); viz_round(v)
    assert v.last_planned_agent == "FEM_Agent" and "FEM_Agent" in v.known_agents

@case
def resume_restores_history_tokens_and_the_worker_name():
    d, v = viz_fixture(); viz_round(v)
    v2 = LiveVisualizer(canvas_obj=CANVAS, explog_obj=EXPLOG); v2.set_working_directory(str(d))
    assert len(v2.steps) == 3 and v2.last_planned_agent == "FEM_Agent"
    assert v2.tokens_total == {"input": 3600, "output": 540}

PASS  each_step_is_attributed_to_the_agent_that_produced_it


PASS  the_worker_name_is_learned_from_the_plan_not_hard_coded


[LiveVisualizer] Restored 3 steps from previous session (next step id: 5)
PASS  resume_restores_history_tokens_and_the_worker_name


## 8. `run_gate` (pause / step-through control)

`status.txt` holds `run`, `stop` or `num`; in `num` mode the append-only grant log
`step_num.txt` decides how many further gate crossings are allowed. **Every crossing is
one step**, and the three agent nodes each cross once per turn. Honored tokens and the
unused budget persist to `step_gate_state.json` so a restart cannot re-grant history.

### 8.1 `parse_tokens` -- the two notations

In [27]:
@case
def bare_digits_are_one_grant_per_character():
    assert G.parse_tokens("1312") == [1, 3, 1, 2]
    assert G.parse_tokens("1212410") == [1, 2, 1, 2, 4, 1, 0]

@case
def separated_numbers_allow_grants_above_nine():
    for text in ("1,3,1,2,10", "1, 3 , 1,2, 10", "1 3 1 2 10", "1;3;1;2;10"):
        assert G.parse_tokens(text) == [1, 3, 1, 2, 10], text

@case
def surrounding_whitespace_and_empties_are_tolerated():
    assert G.parse_tokens("  1312\n") == [1, 3, 1, 2]        # a trailing newline used to defeat "stop"
    assert G.parse_tokens("") == [] and G.parse_tokens("  \n ") == []

@case
def non_numbers_are_rejected():
    for bad in ("12a4", "1,-3", "three", "1.5"):
        try: G.parse_tokens(bad); assert False, bad
        except G.MalformedStepLog: pass

PASS  bare_digits_are_one_grant_per_character
PASS  separated_numbers_allow_grants_above_nine
PASS  surrounding_whitespace_and_empties_are_tolerated
PASS  non_numbers_are_rejected


### 8.2 `reconcile` -- only the appended tokens are granted

In [28]:
@case
def appending_grants_the_new_tokens_only():
    assert G.reconcile([], [1]) == (1, [1], None)                        # fresh run
    assert G.reconcile([1, 3, 1, 2], [1, 3, 1, 2, 4]) == (4, [1, 3, 1, 2, 4], None)
    assert G.reconcile([1, 3], [1, 3, 2, 5]) == (7, [1, 3, 2, 5], None)  # two at once

@case
def an_unchanged_log_grants_nothing():
    assert G.reconcile([1, 3, 1, 2], [1, 3, 1, 2]) == (0, [1, 3, 1, 2], None)

@case
def notation_may_be_switched_as_long_as_the_whole_log_is_rewritten():
    assert G.reconcile(G.parse_tokens("1312"), G.parse_tokens("1,3,1,2,10")) == (10, [1, 3, 1, 2, 10], None)

@case
def a_half_switch_is_refused_instead_of_granting_thousands():
    # "13124" + ",10" re-reads the history as ONE token: 13124 steps. Refuse it.
    grant, seen, warn = G.reconcile([1, 3, 1, 2, 4], G.parse_tokens("13124,10"))
    assert grant == 0 and seen == [13124, 10] and "1,3,1,2,4,10" in warn

@case
def a_rewrite_rebaselines_and_grants_nothing():
    grant, seen, warn = G.reconcile([1, 3, 1, 2], [1, 3, 1, 5])
    assert grant == 0 and seen == [1, 3, 1, 5] and "append-only" in warn
    assert G.reconcile(seen, [1, 3, 1, 5, 2]) == (2, [1, 3, 1, 5, 2], None)   # works again

PASS  appending_grants_the_new_tokens_only
PASS  an_unchanged_log_grants_nothing
PASS  notation_may_be_switched_as_long_as_the_whole_log_is_rewritten
PASS  a_half_switch_is_refused_instead_of_granting_thousands
PASS  a_rewrite_rebaselines_and_grants_nothing


### 8.3 `init_control_files` / `wait_for_turn` -- the live gate

In [29]:
import threading as _th, time as _t

def gate_dir():
    """A fresh working directory with a fast poll, for the blocking cases."""
    d = fresh()
    var.RUN_GATE_POLL_SECONDS, var.RUN_GATE_HEARTBEAT_SECONDS = 0.01, 0.05
    return d

def release_after(d, delay, **files):
    def go():
        _t.sleep(delay)
        for name, text in files.items():
            (Path(d) / f"{name}.txt").write_text(text)
    _th.Thread(target=go, daemon=True).start()

@case
def init_creates_defaults_and_a_resume_never_clobbers_them():
    d = gate_dir(); G.init_control_files(str(d))
    assert (Path(d) / "status.txt").read_text() == "num" and (Path(d) / "step_num.txt").read_text() == "1"
    (Path(d) / "status.txt").write_text("run"); (Path(d) / "step_num.txt").write_text("1312")
    G.init_control_files(str(d))
    assert (Path(d) / "status.txt").read_text() == "run" and (Path(d) / "step_num.txt").read_text() == "1312"

@case
def run_mode_never_blocks_and_spends_no_budget():
    d = gate_dir(); G.init_control_files(str(d), mode="run")
    for _ in range(3): G.wait_for_turn("FEM_Agent", str(d))
    assert not (Path(d) / "step_gate_state.json").exists()

@case
def num_mode_spends_one_step_per_crossing_then_waits_for_a_grant():
    d = gate_dir(); G.init_control_files(str(d))
    G.wait_for_turn("Supervisor", str(d))                      # the initial "1"
    assert G.read_state(str(d)) == {"seen": [1], "remaining": 0}
    release_after(d, 0.05, step_num="13")                      # append a grant of 3
    G.wait_for_turn("FEM_Agent", str(d))
    assert G.read_state(str(d)) == {"seen": [1, 3], "remaining": 2}
    G.wait_for_turn("FEM_Agent", str(d)); G.wait_for_turn("Boss", str(d))
    assert G.read_state(str(d))["remaining"] == 0

@case
def stop_and_unrecognised_values_both_pause():
    for bad in ("stop", "gibberish"):                          # unknown fails SAFE
        d = gate_dir(); G.init_control_files(str(d), mode="run")
        (Path(d) / "status.txt").write_text(bad)
        release_after(d, 0.05, status="run")
        t0 = _t.monotonic(); G.wait_for_turn("Supervisor", str(d))
        assert _t.monotonic() - t0 >= 0.04, bad

@case
def a_restart_does_not_regrant_the_history():
    d = gate_dir(); G.init_control_files(str(d))
    (Path(d) / "step_num.txt").write_text("13124")             # 1+3+1+2+4 = 11
    for _ in range(11): G.wait_for_turn("FEM_Agent", str(d))
    assert G.read_state(str(d)) == {"seen": [1, 3, 1, 2, 4], "remaining": 0}
    G.init_control_files(str(d))                               # simulated restart
    assert G.read_state(str(d)) == {"seen": [1, 3, 1, 2, 4], "remaining": 0}

@case
def a_malformed_log_pauses_and_recovers_when_repaired():
    d = gate_dir(); G.init_control_files(str(d)); G.wait_for_turn("Supervisor", str(d))
    (Path(d) / "step_num.txt").write_text("1oops")
    release_after(d, 0.05, step_num="12")
    G.wait_for_turn("FEM_Agent", str(d))
    assert G.read_state(str(d)) == {"seen": [1, 2], "remaining": 1}

[gate] mode 'num' (edit /tmp/tmpcuz2tcfo/tmph2ke74uf/status.txt to change; append to /tmp/tmpcuz2tcfo/tmph2ke74uf/step_num.txt to grant steps)
[gate] mode 'run' (edit /tmp/tmpcuz2tcfo/tmph2ke74uf/status.txt to change; append to /tmp/tmpcuz2tcfo/tmph2ke74uf/step_num.txt to grant steps)
PASS  init_creates_defaults_and_a_resume_never_clobbers_them
[gate] mode 'run' (edit /tmp/tmpcuz2tcfo/tmpwt4cvm21/status.txt to change; append to /tmp/tmpcuz2tcfo/tmpwt4cvm21/step_num.txt to grant steps)
PASS  run_mode_never_blocks_and_spends_no_budget
[gate] mode 'num' (edit /tmp/tmpcuz2tcfo/tmpr2eiby1i/status.txt to change; append to /tmp/tmpcuz2tcfo/tmpr2eiby1i/step_num.txt to grant steps)
[gate] granted 1 step(s); budget is now 1.
[gate] Supervisor proceeding; 0 step(s) left in the budget.
[gate] FEM_Agent paused: no step budget left. cwd: /tmp/tmpcuz2tcfo/tmpr2eiby1i


[gate] granted 3 step(s); budget is now 3.
[gate] FEM_Agent proceeding; 2 step(s) left in the budget.
[gate] FEM_Agent proceeding; 1 step(s) left in the budget.
[gate] Boss proceeding; 0 step(s) left in the budget.
PASS  num_mode_spends_one_step_per_crossing_then_waits_for_a_grant
[gate] mode 'run' (edit /tmp/tmpcuz2tcfo/tmprlyeazbf/status.txt to change; append to /tmp/tmpcuz2tcfo/tmprlyeazbf/step_num.txt to grant steps)
[gate] Supervisor paused: status is 'stop'. cwd: /tmp/tmpcuz2tcfo/tmprlyeazbf
[gate] mode 'run' (edit /tmp/tmpcuz2tcfo/tmpic96yyz0/status.txt to change; append to /tmp/tmpcuz2tcfo/tmpic96yyz0/step_num.txt to grant steps)
[gate] Supervisor paused: unrecognised status 'gibberish' (expected one of ('run', 'stop', 'num')). cwd: /tmp/tmpcuz2tcfo/tmpic96yyz0


PASS  stop_and_unrecognised_values_both_pause


[gate] mode 'num' (edit /tmp/tmpcuz2tcfo/tmp_9rljgu6/status.txt to change; append to /tmp/tmpcuz2tcfo/tmp_9rljgu6/step_num.txt to grant steps)
[gate] granted 11 step(s); budget is now 11.
[gate] FEM_Agent proceeding; 10 step(s) left in the budget.
[gate] FEM_Agent proceeding; 9 step(s) left in the budget.
[gate] FEM_Agent proceeding; 8 step(s) left in the budget.
[gate] FEM_Agent proceeding; 7 step(s) left in the budget.
[gate] FEM_Agent proceeding; 6 step(s) left in the budget.
[gate] FEM_Agent proceeding; 5 step(s) left in the budget.
[gate] FEM_Agent proceeding; 4 step(s) left in the budget.
[gate] FEM_Agent proceeding; 3 step(s) left in the budget.
[gate] FEM_Agent proceeding; 2 step(s) left in the budget.
[gate] FEM_Agent proceeding; 1 step(s) left in the budget.
[gate] FEM_Agent proceeding; 0 step(s) left in the budget.
[gate] mode 'num' (edit /tmp/tmpcuz2tcfo/tmp_9rljgu6/status.txt to change; append to /tmp/tmpcuz2tcfo/tmp_9rljgu6/step_num.txt to grant steps)
PASS  a_restart_doe

## 9. Summary

In [30]:
df = pd.DataFrame(RESULTS, columns=["case", "ok", "error"])
print(f"{int(df.ok.sum())}/{len(df)} passed")
failed = df[~df.ok]
if len(failed): display(failed)
assert failed.empty, "some cases failed -- see table above"

101/101 passed
